# Ejercicios de Regresión Bayesiana

**Estadística Bayesiana** — Alfredo Sánchez Alberca

In [ ]:
# Configuración inicial
library(tidyverse)
library(rstanarm)
library(bayesplot)
library(tidybayes)
library(bayesrules)
library(broom)
library(knitr)
options(mc.cores = parallel::detectCores())
rstan_options(auto_write = TRUE)

# Carga de datos
df <- read_csv("datos/peso-estatura.csv")
head(df)

## Ejercicio 1 — Ajuste de un modelo de regresión lineal simple

Utilizar la función `stan_glm()` para realizar un ajuste del modelo de regresión del peso sobre el sexo usando el conjunto de datos `peso-estatura` y distribuciones a priori poco informativas para los parámetros del modelo.

- ¿Qué distribuciones a priori propone `stan_glm()` para los parámetros del modelo?
- ¿Qué representan los parámetros del modelo?

> Nota: cuando la variable independiente es categórica, `stan_glm()` crea automáticamente una variable indicadora (dummy) tomando una categoría como referencia. En el caso del sexo, toma `H` como referencia (valor 0) y `M` recibe el valor 1.

In [ ]:
set.seed(123)
modelo_peso_sexo <- stan_glm(... ~ ..., data = df, family = ...,
  prior_intercept = normal(0, 2.5, autoscale = TRUE),
  prior = normal(0, 2.5, autoscale = TRUE),
  prior_aux = exponential(1, autoscale = TRUE),
  chains = 4, iter = 5000 * 2, seed = 123)

In [ ]:
# Distribuciones a priori propuestas por stan_glm()
prior_summary(...)

In [ ]:
# Análisis de las cadenas de Markov
mcmc_trace(..., size = 0.1)

In [ ]:
# Diagnóstico: número efectivo de muestras y R-hat
neff_ratio(...) |> t()
rhat(...) |> t()

## Ejercicio 2 — Intervalos de credibilidad de los parámetros

Obtener los intervalos de credibilidad del 90% de las distribuciones a posteriori de los parámetros ajustados en el modelo de regresión del peso sobre el sexo. ¿Cómo se interpretarían?

In [ ]:
# Intervalos de credibilidad del 90% para cada parámetro
modelo_peso_sexo |>
  ...(`(Intercept)`, sexoM, sigma) |>
  mean_qi(.width = ...)

In [ ]:
# Representación gráfica de los intervalos de credibilidad
mcmc_intervals(modelo_peso_sexo,
  prob = ..., prob_outer = ...,
  point_est = "mean") +
  labs(title = "Intervalos de credibilidad del 90% de los parámetros del modelo") +
  theme_gray()

## Ejercicio 3 — Contraste de hipótesis bayesiano

Realizar un contraste de comparación de medias para ver si el peso medio de los hombres es mayor que el de las mujeres, y calcular la probabilidad de que esto sea cierto a partir de las distribuciones a posteriori de los parámetros del modelo.

$$H_0: \mu_H > \mu_M \iff \beta_1 < 0$$

In [ ]:
# P(β1 < 0) = P(peso_H > peso_M)
modelo_peso_sexo |>
  spread_draws(`(Intercept)`, sexoM) |>
  summarise(prob_H_mayor_M = ...) |>
  kable()

## Ejercicio 4 — Predicciones

Realizar la predicción de la media y de un nuevo valor del peso para un hombre y para una mujer utilizando el modelo de regresión del peso sobre el sexo. Calcular los intervalos de credibilidad del 90% para cada una de las predicciones y representarlas gráficamente sobre el diagrama de dispersión.

In [ ]:
# Nuevos datos: un hombre y una mujer
datos_nuevos_sexo <- tibble(sexo = c("H", "M"))

# Predicción de la media
pred_media_sexo <- ...(modelo_peso_sexo,
  newdata = datos_nuevos_sexo, value = "Predicción_media")
mean_qi(pred_media_sexo, .width = ...)

In [ ]:
# Predicción de un nuevo valor
pred_sexo <- ...(modelo_peso_sexo,
  newdata = datos_nuevos_sexo, value = "Predicción")
mean_qi(pred_sexo, .width = ...)

In [ ]:
# Representación gráfica sobre el diagrama de dispersión
ggplot(df, aes(x = sexo, y = peso)) +
  geom_jitter(color = "steelblue", alpha = 0.4, width = 0.1) +
  stat_pointinterval(
    data = pred_media_sexo,
    aes(x = sexo, y = Predicción_media),
    .width = 0.9, color = "green4", linewidth = 3, size = 5
  ) +
  stat_pointinterval(
    data = pred_sexo,
    aes(x = sexo, y = Predicción),
    .width = 0.9, color = "red", linewidth = 1, size = 3
  ) +
  labs(title = "Predicciones del peso por sexo (verde = media, rojo = nueva observación)",
       x = "Sexo", y = "Peso (kg)") +
  theme_minimal()

## Ejercicio 5 — Métricas de evaluación del modelo

Calcular las métricas de evaluación del modelo de regresión del peso sobre el sexo (`mae`, `mae_scaled`, cobertura de los intervalos de credibilidad y $R^2$ bayesiano). ¿Qué conclusiones se pueden extraer?

In [ ]:
# Error absoluto mediano y cobertura de los intervalos de credibilidad
...(modelo_peso_sexo, data = df, prob_outer = ...)

In [ ]:
# Coeficiente de determinación bayesiano R²
...(modelo_peso_sexo) |>
  mean_qi(.width = 0.9)

## Ejercicio 6 — Evaluación mediante validación cruzada

Realizar la evaluación del modelo de regresión del peso sobre el sexo mediante validación cruzada con 10 pliegues. ¿Qué conclusiones se pueden extraer comparando con las métricas del ejercicio anterior?

In [ ]:
set.seed(123)
eval_cv_sexo <- ...(
  model = modelo_peso_sexo, data = df, k = ..., prob_outer = 0.9)
eval_cv_sexo$cv |> kable()

## Ejercicio 7 — Diseño de un modelo de regresión lineal múltiple

¿Qué modelo de regresión lineal múltiple ajustarías para modelizar el peso de una persona en función de su estatura y su sexo? ¿Qué distribuciones a priori utilizarías para los parámetros del modelo?

Una vez definidas las distribuciones a priori, valídalas simulando rectas de regresión a priori y comprobando que son razonables. Luego ajusta el modelo a los datos.

In [ ]:
# Simulación de las distribuciones a priori (prior_PD = TRUE)
set.seed(123)
modelo_peso_estatura_sexo_prior <- stan_glm(
  peso ~ ... ,
  data = df,
  family = gaussian(),
  prior_intercept = normal(..., ...),
  prior = normal(location = c(..., ...), scale = c(..., ...)),
  prior_aux = exponential(...),
  chains = 4, iter = 5000 * 2, seed = 123,
  prior_PD = TRUE
)

In [ ]:
# Validación visual: 50 rectas de regresión a priori
df |>
  ...(modelo_peso_estatura_sexo_prior, ndraws = 50, value = "Predicción_media") |>
  ggplot(aes(x = estatura, y = peso, color = sexo)) +
  geom_line(aes(y = Predicción_media, group = paste(sexo, .draw)), alpha = 0.2) +
  labs(title = "50 rectas de regresión a priori", x = "Estatura (cm)", y = "Peso (kg)") +
  theme_minimal()

In [ ]:
# Ajuste del modelo a los datos
modelo_peso_estatura_sexo <- ...(modelo_peso_estatura_sexo_prior, prior_PD = FALSE)

In [ ]:
# Intervalos de credibilidad del 90% de los parámetros
modelo_peso_estatura_sexo |>
  ...(`(Intercept)`, estatura, sexoM, sigma) |>
  mean_qi(.width = 0.9)

In [ ]:
# 50 rectas de regresión a posteriori
df |>
  ...(modelo_peso_estatura_sexo, ndraws = 50, value = "Predicción_media") |>
  ggplot(aes(x = estatura, y = peso, color = sexo)) +
  geom_point(data = df, alpha = 0.4) +
  geom_line(aes(y = Predicción_media, group = paste(sexo, .draw)), alpha = 0.2) +
  labs(title = "50 rectas de regresión a posteriori",
       x = "Estatura (cm)", y = "Peso (kg)") +
  theme_minimal()

## Ejercicio 8 — Comparación de modelos mediante ELPD

Comparar la ELPD del modelo de regresión simple del peso sobre el sexo con la ELPD del modelo de regresión múltiple del peso sobre la estatura y el sexo mediante validación cruzada LOO-CV. ¿Qué conclusiones se pueden extraer de esta comparación?

Recuerda la regla de Vehtari para interpretar la diferencia:

| $\Delta\text{ELPD} / \text{SE}$ | Interpretación |
|:---:|:---|
| $< 2$ | Diferencia no sustancial |
| $[2, 4)$ | Diferencia moderada |
| $\geq 4$ | Diferencia sustancial |

In [ ]:
# ELPD del modelo de regresión simple (peso ~ sexo)
loo_sexo <- ...(modelo_peso_sexo)
loo_sexo$estimates[1, ] |> t()

In [ ]:
# ELPD del modelo de regresión múltiple (peso ~ estatura + sexo)
loo_estatura_sexo <- loo(modelo_peso_estatura_sexo)
loo_estatura_sexo$estimates[1, ] |> t()

In [ ]:
# Comparación de los dos modelos
...(loo_sexo, loo_estatura_sexo)[, c("elpd_diff", "se_diff")]

## Ejercicio 9 — Comparación de modelos con y sin interacción

Ajustar un modelo de regresión múltiple del peso sobre la estatura y el sexo **con interacción** entre ambas variables, y comparar su ELPD con la del modelo sin interacción. ¿Hay evidencia de interacción entre la estatura y el sexo?

In [ ]:
# Ajuste del modelo con interacción (peso ~ estatura * sexo)
set.seed(123)
modelo_peso_estatura_sexo_interaccion <- stan_glm(
  peso ~ ... ,
  data = df,
  family = gaussian(),
  prior_intercept = normal(..., ...),
  prior = normal(location = c(..., ..., ...), scale = c(..., ..., ...)),
  prior_aux = exponential(...),
  chains = 4, iter = 5000 * 2, seed = 123
)

In [ ]:
# Intervalos de credibilidad del 90% de los parámetros
modelo_peso_estatura_sexo_interaccion |>
  gather_draws(`(Intercept)`, estatura, sexoM, `estatura:sexoM`, sigma) |>
  mean_qi(.width = 0.9)

In [ ]:
# 50 rectas de regresión a posteriori con interacción
df |>
  add_epred_draws(modelo_peso_estatura_sexo_interaccion, ndraws = 50, value = "Predicción_media") |>
  ggplot(aes(x = estatura, y = peso, color = sexo)) +
  geom_point(data = df, alpha = 0.4) +
  geom_line(aes(y = Predicción_media, group = paste(sexo, .draw)), alpha = 0.2) +
  labs(title = "50 rectas de regresión a posteriori (modelo con interacción)",
       x = "Estatura (cm)", y = "Peso (kg)") +
  theme_minimal()

In [ ]:
# Comparación ELPD: sin interacción vs. con interacción
loo_sin_interaccion <- loo(modelo_peso_estatura_sexo)
loo_con_interaccion <- loo(modelo_peso_estatura_sexo_interaccion)
loo_compare(loo_sin_interaccion, loo_con_interaccion)[, c("elpd_diff", "se_diff")]